In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import os

dir = "/content/drive/MyDrive/HLT project/Project files/sentiment_annotated_with_texts.csv"


with open(dir, 'r') as f:
  sent = pd.read_csv(f, delimiter=',', quotechar='"')



In [3]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from itertools import chain
import nltk
import sklearn
import scipy.stats
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV

! pip install sklearn-crfsuite
import sklearn_crfsuite
from sklearn_crfsuite import scorers
from sklearn_crfsuite import metrics
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.0 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [4]:
all_text = sent[['text', 'true_sentiment']]
all_text['text'] = all_text['text'].apply(lambda d: nltk.word_tokenize(d))
all_text_c = all_text.copy()
all_text_t = all_text.copy()

<ipython-input-4-c407fc17fd8f>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_text['text'] = all_text['text'].apply(lambda d: nltk.word_tokenize(d))


In [5]:
def split_sen(d):
  new_d = []
  sent = []
  for w in d:
    if w == '.':
      sent.append(w)
      sent = []
      new_d.append(sent)
    else:
      sent.append(w)
  return new_d

#all_text_c['text'] = all_text['text'].apply(lambda d: split_sen(d))


In [6]:

all_text_t['text'] = all_text_c['text'].apply(lambda d: nltk.pos_tag(d))

In [7]:
all_text_t[:4]

,text,true_sentiment
0,"[(The, DT), (Euro, NNP), (was, VBD), (able, JJ...",Positive
1,"[(EUR/CHF, NN), (yesterday, NN), (broke, VBD),...",Positive
2,"[(EUR/CHF, NNP), (vaults, NNS), (parity, NN), ...",Neutral
3,"[(EUR/CHF, NNP), (climbs, VBZ), (back, RB), (a...",Positive


In [8]:
txt_sent = "/content/drive/MyDrive/HLT project/Project files/NRC-VAD-Lexicon.txt"

data = {}

with open(txt_sent, "r") as f:
  t = f.read()
  for si in t.split('\n'):
    si_w = (si +'\t').split('\t')
    if si_w[0] != '':
      data[si_w[0]] = list(map(lambda fl: float(fl), si_w[1:-1]))
    else:
      pass

In [9]:
import re

txt_sent = "/content/drive/MyDrive/HLT project/Project files/SentiWordNet_3.0.0.txt"
senti_word = {}
regex = '0(\.\d+)?\s0(\.\d+)?\s([a-z_\-]+#\d+\s)+'
pattern = re.compile(regex)


with open(txt_sent, "r") as f:
  t = f.read()
  for si in t.split("\n"):
    if len(si) > 0 and si[0] != '#':
      matched = pattern.search(si)
      if matched:
        val = matched.group(0).split('\t')
        scor = [float(val[0]), float(val[1])]
        wors = re.split('#\d+\s?', val[2])
        wors.pop(-1)
        for wr in wors:
          senti_word[wr] = scor

df_senti = pd.DataFrame(senti_word, index=["pos_score", "negative_score"])




In [10]:

for w,v in data.items():
  if len(v) < 3:
    print(w)
df2 = pd.DataFrame(data, index=['valence', 'arousal', 'dominance'])


In [11]:
import string

def word2features(sent, i):
    word = sent[i][0]
    postag = sent[i][1]

    regex= "^\+|-\d+\.\d+"
    pattern = re.compile(regex)

    add_score = False




    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'len(word)': len(word),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'postag': postag,
        'postag[:2]': postag[:2],
        'word.stemmed': re.sub(r'(.{2,}?)([aeiougyn]+$)',r'\1', word.lower()),
        'word.ispunctuation': (word in string.punctuation),
    }
    try:
      sc = df_senti[word.lower()]
      features.update({
              'word.posScore': sc[0],
              'word.negScore': sc[1],
          })
      add_score = True
    except KeyError:
      pass

    if pattern.match(word) is not None:
      features['interests'] = word
    else:
      features['interests'] = 0

    if i > 0:
        word1 = sent[i-1][0]
        postag1 = sent[i-1][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
        })
        com_wrod = word1.lower()+ '_'+word.lower()
        try:
          sc = df_senti[com_wrod]
          add_score = True
          features.update({
              '-1:word.posScore': sc[0],
              '-1:word.negScore': sc[1],
          })
        except KeyError:
          pass
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        word1 = sent[i+1][0]
        postag1 = sent[i+1][1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
        })

        com_wrod = word1.lower()+ '_'+word.lower()
        try:
          sc = df_senti[com_wrod]
          add_score = True
          features.update({
              '+1:word.posScore': sc[0],
              '+1:word.negScore': sc[1],
          })
        except KeyError:
          pass
    else:
        features['EOS'] = True

    try:

      features['lexicon'] = df2[word].sum()/3.0
    except KeyError:
      pass

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


def sent2labels(sent):
    return [label for token, postag, label in sent]

def sent2tokens(sent):
    return [token for token, postag, label in sent]

In [12]:
all_X = [sent2features(s) for s in all_text_t['text']]


In [13]:
import numpy as np

all_Y = [list(np.repeat(l, len(all_X[i]))) for i,l in enumerate(all_text_t['true_sentiment'])]

In [14]:
all_l = len(all_X)

train_ind = int(all_l * 0.9)
X_train = all_X[0:train_ind]
X_test = all_X[train_ind:all_l]

y_train = all_Y[0:train_ind]
y_test = all_Y[train_ind:all_l]



In [15]:
%%time
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.2,
    c2=0.2,
    max_iterations=300,
    all_possible_transitions=True,
    verbose=True
)
try:
    crf.fit(X_train, y_train)
except AttributeError:
    pass

loading training data to CRFsuite: 100%|██████████| 2061/2061 [00:17<00:00, 117.11it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 136294
Seconds required: 3.001

L-BFGS optimization
c1: 0.200000
c2: 0.200000
num_memories: 6
max_iterations: 300
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=1.04  loss=564234.61 active=127092 feature_norm=1.00
Iter 2   time=0.53  loss=408646.49 active=134347 feature_norm=1.39
Iter 3   time=0.54  loss=283371.68 active=133158 feature_norm=1.85
Iter 4   time=1.06  loss=226338.02 active=134323 feature_norm=3.22
Iter 5   time=0.74  loss=189678.64 active=134664 feature_norm=3.81
Iter 6   time=0.79  loss=183012.39 active=136249 feature_norm=3.91
Iter 7   time=0.76  loss=99466.28 active=135213 feature_norm=4.14
Iter 8   time=0.81  loss=86979.06 active=135802 feature_norm=4.36
Iter 9   time=0.81  loss=69251.67 active=135508 feature_norm=4.5

In [16]:
labels = list(crf.classes_)
labels

['Positive', 'Neutral', 'Negative']

In [17]:
y_pred = crf.predict(X_test)


In [18]:
metrics.flat_f1_score(y_test, y_pred,
                      average='weighted', labels=labels)

0.4716911259834994

In [19]:
print(crf.)

1668004


In [20]:
from sklearn_crfsuite.utils import flatten
from sklearn import metrics as skmetrics

sorted_labels = sorted(
    labels,
    key=lambda name: (name[1:], name[0])
)
y_test_flat = flatten(y_test)
y_pred_flat = flatten(y_pred)
print(skmetrics.classification_report(y_test_flat, y_pred_flat, labels=sorted_labels, digits=3))

              precision    recall  f1-score   support

    Negative      0.402     0.533     0.458     20380
     Neutral      0.471     0.433     0.451     26633
    Positive      0.542     0.462     0.499     30559

    accuracy                          0.471     77572
   macro avg      0.472     0.476     0.469     77572
weighted avg      0.481     0.471     0.472     77572

